<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/16_GES_Aware_Genomic_RAG_Cell_7C9_Protocol_Amendment_A003_Hybrid_Single_Reviewer_Evaluation_Freeze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact Cell 7C8 lineage, and fail-closed output paths

In [ ]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re
import tempfile

import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '16_GES_Aware_Genomic_RAG_Cell_7C9_'
    'Protocol_Amendment_A003_Hybrid_Single_Reviewer_Evaluation_Freeze.ipynb'
)
CELL_ID = '7C9'
STAGE = '7C'
AMENDMENT_ID = 'A003'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
REPEAT_ITEMS_PER_QUESTION = 2
EXPECTED_REPEAT_ITEMS = EXPECTED_QUESTIONS * REPEAT_ITEMS_PER_QUESTION
MINIMUM_WASHOUT_DAYS = 14

EXPECTED_CELL_7C8_TERMINAL_DECISION = (
    'PASS_STAGE7C8_1440_OPAQUE_BLINDED_REVIEW_ITEMS_2880_TWO_REVIEWER_ASSIGNMENTS_'
    '23040_RUBRIC_SCORING_ROWS_AND_ATOMIC_CLAIM_TEMPLATES_MATERIALIZED_WITH_INTERNAL_'
    'ROUTING_SEPARATED_CHECKSUM_PROTECTED_NO_CONDITION_UNBLINDING_SCORE_BEARING_'
    'ARTIFACTS_RUN_AGGREGATION_RAG_METRICS_BOOTSTRAP_OR_ARM_COMPARISON_HUMAN_'
    'BLINDED_REVIEW_REQUIRED_NEXT_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------------------
# Exact successful Cell 7C8 package from the terminal PASS.
# --------------------------------------------------------------------------------------------------
CELL_7C8_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)
CELL_7C8_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)
CELL_7C8_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c8_blinded_reviewer_packet_v1'
)

CELL_7C8 = OrderedDict([
    ('review_packet', {
        'path': CELL_7C8_EXEC_DIR / 'cell_7c8_blinded_review_packet_v1.parquet',
        'sha256': '232ea13fb2e09fa8821bc61fa8c4bd9d7f04f8c30eb6b47360ed8ee1f49d8466',
    }),
    ('reviewer_assignments', {
        'path': CELL_7C8_EXEC_DIR / 'cell_7c8_two_reviewer_assignment_inventory_v1.csv',
        'sha256': '5ea140d587e704a896942b610447f4a4f93fda023cb075f06f35f8a3120fb899',
    }),
    ('rubric_scoring_template', {
        'path': CELL_7C8_EXEC_DIR / 'cell_7c8_reviewer_rubric_scoring_template_v1.csv',
        'sha256': '05fe6dfbe30d02f93e35d5798a4f22fca18c5a5e0f91346b79455b891f851d09',
    }),
    ('atomic_claim_template', {
        'path': CELL_7C8_EXEC_DIR / 'cell_7c8_reviewer_atomic_claim_annotation_template_v1.csv',
        'sha256': '4d38d6eacae38f4c3cc217917c16342ea3f001ccb952e64a6aef37d0f1da679e',
    }),
    ('adjudicator_template', {
        'path': CELL_7C8_EXEC_DIR / 'cell_7c8_third_adjudicator_disagreement_template_v1.csv',
        'sha256': '9177618b9bbf85d0baba404e25630cd96aa40a920d4b2c478603a8a1dfd1cb99',
    }),
    ('internal_routing_map', {
        'path': CELL_7C8_CONFIG_DIR / 'cell_7c8_internal_blinded_review_routing_map_v1.parquet',
        'sha256': '8c65375e6a24761bd14e2d59d837507c67146ed88e8e64a533bca304598c696c',
    }),
    ('reviewer_instructions', {
        'path': CELL_7C8_CONFIG_DIR / 'cell_7c8_blinded_reviewer_instructions_v1.json',
        'sha256': '268ac9570ea0ae7a78a72cc5537c3f31a79d796880460c93b9fe925fdd43016e',
    }),
    ('input_inventory', {
        'path': CELL_7C8_CONFIG_DIR / 'cell_7c8_verified_input_inventory_v1.csv',
        'sha256': '0f4c4b0ff87b0431e82bfbfccdccf77521703dc13f487d049e3ee5ff4b3a9741',
    }),
    ('execution_report', {
        'path': CELL_7C8_QC_DIR / 'cell_7c8_blinded_reviewer_packet_execution_report_v1.json',
        'sha256': '71973d16c6befbbf70459068e621895e939fc98bd5c6503cb22f6972ce24fb78',
    }),
    ('qc', {
        'path': CELL_7C8_QC_DIR / 'cell_7c8_blinded_reviewer_packet_qc_v1.json',
        'sha256': 'dfd0497e15b64e64e5355a764e5ab84cda5dbc7c13eaf40ddf0d438450e92f3e',
    }),
    ('manifest', {
        'path': CELL_7C8_CONFIG_DIR / 'cell_7c8_blinded_reviewer_packet_manifest_v1.json',
        'sha256': 'e8a2bf79d7c0479063af0fbfb73a77427746c79bdc11600672a82af678b25af7',
    }),
])

# Exact upstream structured answer key is needed only for lineage and next-cell authorization.
ANSWER_KEY_SHA256 = 'bccf3691336ed8e21a4e7b2876fcf6883ee9a8447938a0edf61d41fc5e0d0332'
RUBRIC_SHA256 = '4185ad173c5e3e2ecea31268b8d833ba111839f5ddd1b96c3fda083e01afcba9'
CONTEXT_INVENTORY_SHA256 = 'd23b604dac645c158cd1b95cc9cd564fb6b9447279756b6cae589b5572dd4623'

AMEND_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1'
)

OUTPUTS = OrderedDict([
    ('amendment',
     AMEND_DIR / 'protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1.json'),
    ('deterministic_scoring_spec',
     AMEND_DIR / 'protocol_amendment_A003_deterministic_scoring_spec_v1.json'),
    ('human_review_spec',
     AMEND_DIR / 'protocol_amendment_A003_single_blinded_human_review_spec_v1.json'),
    ('repeat_assessment_spec',
     AMEND_DIR / 'protocol_amendment_A003_intrarater_repeat_assessment_spec_v1.json'),
    ('input_inventory',
     AMEND_DIR / 'protocol_amendment_A003_verified_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'protocol_amendment_A003_qc_v1.json'),
    ('manifest',
     AMEND_DIR / 'protocol_amendment_A003_manifest_v1.json'),
])

for directory in (AMEND_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C9 / A003 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Amendment directory: {AMEND_DIR}')
print(f'QC directory       : {QC_DIR}')

Amendment directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1
QC directory       : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/protocol_amendment_A003_hybrid_single_reviewer_evaluation_v1


## 2. SHA-256, sidecar, serialization, and metadata helpers

In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\\n'
            f'Expected: {expected_sha256}\\nObserved: {observed}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing SHA-256 sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def csv_metadata(path: Path) -> dict[str, Any]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        row_count = sum(1 for _ in reader)
    return {
        'rows': int(row_count),
        'columns': int(len(header)),
        'schema_names': header,
    }


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(
        f'{digest}  {path.name}' + chr(10),
        encoding='utf-8',
    )


with tempfile.TemporaryDirectory(prefix='cell_7c9_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify the complete Cell 7C8 package without opening condition identity

In [ ]:
verified_inputs = []

for artifact_id, spec in CELL_7C8.items():
    record = verify_exact_artifact(
        f'cell_7c8_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C8'
    verified_inputs.append(record)

manifest_7c8 = load_json(CELL_7C8['manifest']['path'])
qc_7c8 = load_json(CELL_7C8['qc']['path'])
report_7c8 = load_json(CELL_7C8['execution_report']['path'])

if manifest_7c8.get('terminal_decision') != EXPECTED_CELL_7C8_TERMINAL_DECISION:
    raise AssertionError('Cell 7C8 terminal PASS mismatch.')
if manifest_7c8.get('next_authorized_cell') is not None:
    raise AssertionError('Cell 7C8 unexpectedly authorizes a downstream cell.')
if manifest_7c8.get('condition_unblinding_authorized') is not False:
    raise AssertionError('Cell 7C8 unexpectedly authorizes condition unblinding.')
if manifest_7c8.get('scientific_metric_calculation_authorized') is not False:
    raise AssertionError('Cell 7C8 unexpectedly authorizes scientific metric calculation.')
if int(qc_7c8.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C8 QC does not report zero failures.')

review_meta = parquet_metadata(CELL_7C8['review_packet']['path'])
routing_meta = parquet_metadata(CELL_7C8['internal_routing_map']['path'])
rubric_meta = csv_metadata(CELL_7C8['rubric_scoring_template']['path'])
claim_meta = csv_metadata(CELL_7C8['atomic_claim_template']['path'])

if review_meta['rows'] != EXPECTED_RESPONSES:
    raise AssertionError(f'Expected 1,440 review items; observed {review_meta["rows"]}.')
if routing_meta['rows'] != EXPECTED_RESPONSES:
    raise AssertionError(f'Expected 1,440 routing rows; observed {routing_meta["rows"]}.')
if rubric_meta['rows'] != 23_040:
    raise AssertionError(f'Expected 23,040 original two-reviewer rubric rows; observed {rubric_meta["rows"]}.')
if claim_meta['rows'] != 2_880:
    raise AssertionError(f'Expected 2,880 original two-reviewer claim rows; observed {claim_meta["rows"]}.')

# Metadata-only leakage check on the reviewer packet.
for prohibited in [
    'blinded_alias',
    'run_id',
    'generation_request_id',
    'condition_id',
    'condition_name',
]:
    if prohibited in review_meta['schema_names']:
        raise AssertionError(f'Cell 7C8 reviewer packet exposes prohibited field: {prohibited}')

print('Cell 7C8 frozen package                : 11/11 exact hashes + sidecars')
print('Cell 7C8 terminal PASS                 : VERIFIED')
print(f'Opaque review items                    : {review_meta["rows"]:,}')
print('Cell 7C8 condition identity             : NOT OPENED')
print('Cell 7C8 score-bearing artifacts        : NOT OPENED')
print('Scientific performance metrics          : NOT CALCULATED')

Cell 7C8 frozen package                : 11/11 exact hashes + sidecars
Cell 7C8 terminal PASS                 : VERIFIED
Opaque review items                    : 1,440
Cell 7C8 condition identity             : NOT OPENED
Cell 7C8 score-bearing artifacts        : NOT OPENED
Scientific performance metrics          : NOT CALCULATED


## 4. Freeze deterministic scoring specification

In [ ]:
deterministic_scoring_spec = {
    'amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'population': {
        'frozen_response_count': EXPECTED_RESPONSES,
        'apply_to_all_responses': True,
    },
    'allowed_inputs_in_cell_7c10': {
        'cell_7c8_review_packet_sha256': CELL_7C8['review_packet']['sha256'],
        'cell_7c8_internal_routing_map_sha256': CELL_7C8['internal_routing_map']['sha256'],
        'structured_answer_key_sha256': ANSWER_KEY_SHA256,
        'score_blind_context_inventory_sha256': CONTEXT_INVENTORY_SHA256,
    },
    'deterministic_metrics': [
        {
            'metric_id': 'evidence_id_context_validity',
            'definition':
                'For every cited evidence_id, test exact membership in the five frozen packet_ids supplied to that response.',
            'outputs': [
                'cited_evidence_id_count',
                'valid_context_evidence_id_count',
                'invalid_or_hallucinated_evidence_id_count',
                'all_cited_evidence_ids_valid',
            ],
            'semantic_support_inferred': False,
        },
        {
            'metric_id': 'evidence_context_coverage',
            'definition':
                'Count distinct supplied context packet_ids cited by the model divided by five; this is descriptive coverage, not citation correctness.',
            'semantic_support_inferred': False,
        },
        {
            'metric_id': 'required_caution_behavior',
            'definition':
                'When the frozen answer key has expected_abstention_or_qualification_required=true, '
                'record whether response_policy is cautious_answer or abstain.',
            'outputs': [
                'expected_caution_required',
                'observed_cautious_or_abstain',
                'required_caution_compliant',
            ],
            'primary_endpoint': False,
        },
        {
            'metric_id': 'unnecessary_abstention_descriptive',
            'definition':
                'When expected_abstention_or_qualification_required=false, record whether response_policy=abstain. '
                'This is a descriptive secondary diagnostic and is not treated as automatically incorrect.',
            'primary_endpoint': False,
        },
        {
            'metric_id': 'structured_output_integrity',
            'definition':
                'Reconfirm presence/type validity of the eight frozen response fields and evidence_ids uniqueness. '
                'This is execution/evaluation QC, not a condition-performance endpoint.',
            'primary_endpoint': False,
        },
    ],
    'explicitly_not_deterministically_scored': [
        'semantic correctness of narrative factual claims',
        'whether a citation truly supports a claim',
        'clinical appropriateness of a qualification',
        'rubric dimensions requiring scientific judgment',
    ],
    'condition_identity_access': False,
    'arm_comparison_allowed': False,
    'run_aggregation_allowed': False,
    'bootstrap_allowed': False,
}

stable_write_json(OUTPUTS['deterministic_scoring_spec'], deterministic_scoring_spec)
write_sidecar(OUTPUTS['deterministic_scoring_spec'])

print('Deterministic scoring population       : 1,440 / 1,440 responses')
print('Evidence-ID membership checks           : FROZEN')
print('Required-caution diagnostic             : FROZEN')
print('Semantic citation support               : HUMAN REVIEW ONLY')
print('Condition identity access               : PROHIBITED')

Deterministic scoring population       : 1,440 / 1,440 responses
Evidence-ID membership checks           : FROZEN
Required-caution diagnostic             : FROZEN
Semantic citation support               : HUMAN REVIEW ONLY
Condition identity access               : PROHIBITED


## 5. Freeze single blinded human-review specification

In [ ]:
human_review_spec = {
    'amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'review_population': {
        'review_all_frozen_responses': True,
        'response_count': EXPECTED_RESPONSES,
        'reviewer_count': 1,
        'reviewer_role_label': 'SINGLE-BLINDED-REVIEWER',
    },
    'reviewer_blinding': {
        'use_opaque_review_item_id': True,
        'hide_blinded_alias': True,
        'hide_A_to_F_condition_identity': True,
        'hide_run_id': True,
        'hide_generation_request_id': True,
        'hide_GES_scores': True,
        'hide_metadata_scores': True,
        'hide_quality_rank': True,
        'hide_semantic_rank_and_score': True,
        'hide_RRF_score': True,
    },
    'human_scoring_scope': {
        'atomic_factual_claims': True,
        'atomic_claim_correctness': True,
        'citation_support_for_each_claim': True,
        'combined_correct_and_citation_supported_label': True,
        'frozen_rubric_dimensions': True,
    },
    'primary_endpoint': {
        'unchanged_from_frozen_design': True,
        'definition':
            'fraction of atomic factual claims that are both correct and citation-supported',
        'calculation_deferred_until_after_review_freeze': True,
    },
    'reviewer_process': {
        'first_pass_judgments_must_be_frozen_before_repeat_assessment': True,
        'reviewer_must_not_access_internal_routing_map': True,
        'reviewer_must_not_access_condition_mapping': True,
        'reviewer_must_not_access_arm_level_results': True,
        'reviewer_must_not_modify_model_response_text': True,
        'reviewer_must_not_modify_frozen_answer_key': True,
        'reviewer_must_not_modify_frozen_context_bundle': True,
    },
    'original_two_reviewer_plan': {
        'status': 'superseded_for_execution_by_protocol_amendment_A003',
        'cell_7c8_two_reviewer_artifacts_preserved': True,
        'reason':
            'Independent reviewer panel unavailable for the present single-author study; '
            'independence will not be simulated or misrepresented.',
    },
    'limitation_to_report':
        'Claim-level human evaluation was performed by one blinded reviewer rather than multiple independent reviewers.',
}

stable_write_json(OUTPUTS['human_review_spec'], human_review_spec)
write_sidecar(OUTPUTS['human_review_spec'])

print('Human reviewers                         : 1')
print('Human review population                 : all 1,440 responses')
print('Condition / alias / run identity         : HIDDEN')
print('Primary endpoint definition              : UNCHANGED')
print('Original two-reviewer artifacts          : PRESERVED, SUPERSEDED FOR EXECUTION')

Human reviewers                         : 1
Human review population                 : all 1,440 responses
Condition / alias / run identity         : HIDDEN
Primary endpoint definition              : UNCHANGED
Original two-reviewer artifacts          : PRESERVED, SUPERSEDED FOR EXECUTION


## 6. Freeze blinded intra-rater repeat-assessment specification

In [ ]:
repeat_assessment_spec = {
    'amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'purpose':
        'Quantify intra-rater consistency without pretending to have independent reviewers.',
    'sampling': {
        'unit': 'Cell 7C8 opaque review_item_id',
        'stratification': 'question_id only',
        'primary_questions': EXPECTED_QUESTIONS,
        'items_per_question': REPEAT_ITEMS_PER_QUESTION,
        'total_repeat_items': EXPECTED_REPEAT_ITEMS,
        'selection_key':
            'For each question_id, compute SHA256("A003_REPEAT|" + question_id + "|" + review_item_id) '
            'and select the two lexicographically smallest hashes.',
        'condition_mapping_used_for_selection': False,
        'run_id_used_for_selection': False,
        'GES_or_metadata_score_used_for_selection': False,
    },
    'reblinding': {
        'repeat_id_rule':
            'RPT-' + 'first24uppercasehex(SHA256("A003_REBLIND|" + review_item_id))',
        'original_review_item_id_hidden_during_repeat': True,
        'first_pass_scores_hidden_during_repeat': True,
        'first_pass_notes_hidden_during_repeat': True,
        'condition_identity_hidden_during_repeat': True,
    },
    'timing': {
        'minimum_washout_days_after_first_pass_completion': MINIMUM_WASHOUT_DAYS,
        'repeat_before_condition_unblinding': True,
        'repeat_before_arm_level_performance_calculation': True,
    },
    'claim_unit_alignment': {
        'repeat_uses_first_pass_frozen_atomic_claim_units': True,
        'prior_correctness_and_citation_support_labels_hidden': True,
        'reason':
            'Intra-rater reliability must compare repeated judgments on identical claim units rather than '
            'confounding judgment consistency with a second claim-segmentation process.',
    },
    'prespecified_reliability_outputs': [
        'raw percent agreement for atomic-claim correctness',
        'Cohen kappa for atomic-claim correctness when estimable',
        'raw percent agreement for citation-supported judgment',
        'Cohen kappa for citation-supported judgment when estimable',
        'raw percent agreement for combined correct-and-citation-supported judgment',
        'Cohen kappa for combined correct-and-citation-supported judgment when estimable',
        'rubric agreement summaries using the frozen rubric response scale; weighted kappa only when the rubric scale is ordered and valid for that calculation',
    ],
    'no_reliability_result_calculated_in_cell_7c9': True,
}

stable_write_json(OUTPUTS['repeat_assessment_spec'], repeat_assessment_spec)
write_sidecar(OUTPUTS['repeat_assessment_spec'])

print(f'Repeat sample                           : {EXPECTED_REPEAT_ITEMS} responses')
print(f'Sampling rule                           : {REPEAT_ITEMS_PER_QUESTION} per question × {EXPECTED_QUESTIONS} questions')
print(f'Minimum washout                         : {MINIMUM_WASHOUT_DAYS} days')
print('Repeat condition identity               : HIDDEN')
print('Reliability statistics                  : PRESPECIFIED, NOT CALCULATED')

Repeat sample                           : 160 responses
Sampling rule                           : 2 per question × 80 questions
Minimum washout                         : 14 days
Repeat condition identity               : HIDDEN
Reliability statistics                  : PRESPECIFIED, NOT CALCULATED


## 7. Freeze Amendment A003 authorization for Cell 7C10

In [ ]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C10_HYBRID_SINGLE_REVIEWER_EVALUATION_PACKET_'
    'MATERIALIZATION_ONLY_FROM_FROZEN_CELL7C8_PACKAGE_WITH_DETERMINISTIC_SCORING_'
    'SPEC_SINGLE_BLINDED_REVIEWER_ALL_1440_RESPONSES_AND_160_RESPONSE_14DAY_'
    'INTRARATER_REPEAT_SAMPLE_NO_CONDITION_UNBLINDING_ARM_COMPARISON_RUN_AGGREGATION_'
    'PRIMARY_ENDPOINT_CALCULATION_OR_BOOTSTRAP'
)

amendment_payload = {
    'protocol_amendment_id': AMENDMENT_ID,
    'cell_id': CELL_ID,
    'stage': STAGE,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'amendment_status': 'PROSPECTIVE_RELATIVE_TO_HUMAN_SCORING',
    'reason_for_amendment':
        'The original two-independent-reviewer plus third-adjudicator plan is not operationally feasible '
        'for the current single-author study because an independent reviewer panel is unavailable. '
        'The study will not simulate reviewer independence. A transparent hybrid evaluation design is frozen '
        'before any response-level human scoring or condition-level performance inspection.',
    'unchanged_upstream_experiment': {
        'questions': 80,
        'conditions': 6,
        'runs_per_question_condition': 3,
        'frozen_llm_responses': 1440,
        'retrieval_changed': False,
        'reranking_changed': False,
        'top5_context_changed': False,
        'prompts_changed': False,
        'llm_model_changed': False,
        'generation_parameters_changed': False,
        'answer_keys_changed': False,
        'rubric_definitions_changed': False,
        'primary_endpoint_definition_changed': False,
    },
    'revised_human_evaluation': {
        'independent_reviewer_count': 1,
        'reviewer_blinded': True,
        'review_all_responses': True,
        'repeat_sample_size': EXPECTED_REPEAT_ITEMS,
        'minimum_washout_days': MINIMUM_WASHOUT_DAYS,
        'intra_rater_reliability_prespecified': True,
    },
    'deterministic_evaluation': {
        'apply_to_all_1440': True,
        'spec_sha256': sha256_file(OUTPUTS['deterministic_scoring_spec']),
    },
    'human_review': {
        'spec_sha256': sha256_file(OUTPUTS['human_review_spec']),
    },
    'repeat_assessment': {
        'spec_sha256': sha256_file(OUTPUTS['repeat_assessment_spec']),
    },
    'authorization_decision': authorization_decision,
    'next_authorized_cell': '7C10',
    'cell_7c10_scope':
        'Materialize the hybrid evaluation packet, deterministic scoring input table, one-reviewer templates, '
        'and frozen 160-item repeat sample/reblinding map. Cell 7C10 may not calculate condition-level scientific '
        'performance or unblind experimental arms.',
    'condition_unblinding_authorized': False,
    'score_bearing_artifact_access_authorized': False,
    'run_aggregation_authorized': False,
    'primary_endpoint_calculation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
    'llm_call_authorized': False,
}

stable_write_json(OUTPUTS['amendment'], amendment_payload)
write_sidecar(OUTPUTS['amendment'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

prewrite_checks = OrderedDict([
    ('cell7c8_all_11_artifacts_verified', len(verified_inputs) == 11),
    ('cell7c8_terminal_pass_exact',
     manifest_7c8.get('terminal_decision') == EXPECTED_CELL_7C8_TERMINAL_DECISION),
    ('review_items_1440', review_meta['rows'] == 1440),
    ('routing_rows_1440', routing_meta['rows'] == 1440),
    ('reviewer_packet_alias_hidden', 'blinded_alias' not in review_meta['schema_names']),
    ('reviewer_packet_run_hidden', 'run_id' not in review_meta['schema_names']),
    ('single_reviewer_count_1',
     human_review_spec['review_population']['reviewer_count'] == 1),
    ('review_all_1440',
     human_review_spec['review_population']['response_count'] == 1440),
    ('repeat_sample_160',
     repeat_assessment_spec['sampling']['total_repeat_items'] == 160),
    ('repeat_2_per_question',
     repeat_assessment_spec['sampling']['items_per_question'] == 2),
    ('washout_14_days',
     repeat_assessment_spec['timing']['minimum_washout_days_after_first_pass_completion'] == 14),
    ('primary_endpoint_unchanged',
     human_review_spec['primary_endpoint']['unchanged_from_frozen_design'] is True),
    ('condition_unblinding_false',
     amendment_payload['condition_unblinding_authorized'] is False),
    ('run_aggregation_false',
     amendment_payload['run_aggregation_authorized'] is False),
    ('primary_endpoint_calc_false',
     amendment_payload['primary_endpoint_calculation_authorized'] is False),
    ('bootstrap_false',
     amendment_payload['bootstrap_inference_authorized'] is False),
    ('arm_comparison_false',
     amendment_payload['arm_comparison_authorized'] is False),
    ('llm_call_false',
     amendment_payload['llm_call_authorized'] is False),
    ('no_human_scoring_in_7c9', True),
    ('no_condition_performance_inspection_in_7c9', True),
])

failed_prewrite = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed_prewrite:
    raise RuntimeError(
        'Cell 7C9 / Amendment A003 prewrite QC failed:\\n- '
        + '\\n- '.join(failed_prewrite)
    )

terminal_decision = (
    'PASS_STAGE7C9_PROTOCOL_AMENDMENT_A003_HYBRID_SINGLE_REVIEWER_EVALUATION_'
    'FROZEN_BEFORE_HUMAN_SCORING_DETERMINISTIC_ALL1440_SINGLE_BLINDED_HUMAN_'
    'REVIEW_ALL1440_AND_160_ITEM_14DAY_INTRARATER_REPEAT_PRESPECIFIED_CELL7C10_'
    'PACKET_MATERIALIZATION_ONLY_AUTHORIZED_NO_CONDITION_UNBLINDING_RUN_AGGREGATION_'
    'PRIMARY_ENDPOINT_CALCULATION_BOOTSTRAP_OR_ARM_COMPARISON'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c8_manifest_sha256': CELL_7C8['manifest']['sha256'],
        'cell_7c8_review_packet_sha256': CELL_7C8['review_packet']['sha256'],
        'cell_7c8_internal_routing_map_sha256': CELL_7C8['internal_routing_map']['sha256'],
        'structured_answer_key_sha256': ANSWER_KEY_SHA256,
        'rubric_sha256': RUBRIC_SHA256,
        'score_blind_context_inventory_sha256': CONTEXT_INVENTORY_SHA256,
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C10',
    'condition_unblinding_authorized': False,
    'run_aggregation_authorized': False,
    'primary_endpoint_calculation_authorized': False,
    'bootstrap_inference_authorized': False,
    'arm_comparison_authorized': False,
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Final readback.
for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Final Cell 7C9 artifact readback failed: {path}')

amend_rb = load_json(OUTPUTS['amendment'])
qc_rb = load_json(OUTPUTS['qc'])
manifest_rb = load_json(OUTPUTS['manifest'])
repeat_rb = load_json(OUTPUTS['repeat_assessment_spec'])

readback_checks = OrderedDict([
    ('amendment_A003_exact', amend_rb['protocol_amendment_id'] == 'A003'),
    ('next_cell_7c10', manifest_rb.get('next_authorized_cell') == '7C10'),
    ('repeat_total_160', repeat_rb['sampling']['total_repeat_items'] == 160),
    ('washout_14', repeat_rb['timing']['minimum_washout_days_after_first_pass_completion'] == 14),
    ('condition_unblinding_false', manifest_rb['condition_unblinding_authorized'] is False),
    ('run_aggregation_false', manifest_rb['run_aggregation_authorized'] is False),
    ('primary_endpoint_false', manifest_rb['primary_endpoint_calculation_authorized'] is False),
    ('bootstrap_false', manifest_rb['bootstrap_inference_authorized'] is False),
    ('arm_comparison_false', manifest_rb['arm_comparison_authorized'] is False),
    ('qc_zero_failures', int(qc_rb['failed_checks']) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C9 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C9')
print('PROTOCOL AMENDMENT A003 — HYBRID SINGLE-REVIEWER EVALUATION DESIGN FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM CELL 7C8 REVERIFICATION')
print(f'Cell 7C8 manifest SHA-256                     : {CELL_7C8["manifest"]["sha256"]}')
print('Cell 7C8 terminal PASS verified               : YES')
print('Cell 7C8 frozen artifacts                     : 11/11 exact hashes + sidecars')
print('Frozen response/review items                  : 1,440')
print('Condition identity opened                     : NO')

print('\\nPROTOCOL AMENDMENT A003')
print('Original human-review plan                    : 2 independent reviewers + third adjudicator')
print('Revised execution plan                        : 1 blinded reviewer + deterministic scoring + intra-rater repeat')
print('Reason                                        : independent reviewer panel unavailable; independence will not be simulated')
print('Human scoring performed in Cell 7C9           : NO')
print('Primary endpoint definition changed           : NO')

print('\\nHYBRID EVALUATION FREEZE')
print('Deterministic scoring                         : all 1,440 responses')
print('Single blinded human review                   : all 1,440 responses')
print('Intra-rater repeat sample                     : 160 responses')
print('Repeat sampling                               : 2 opaque items per each of 80 questions')
print('Minimum repeat washout                        : 14 days')
print('Repeat reblinding                             : REQUIRED')
print('First-pass labels visible during repeat       : NO')

print('\\nCELL 7C10 AUTHORIZATION')
print('Hybrid packet materialization                 : AUTHORIZED')
print('Single-reviewer templates                     : AUTHORIZED')
print('160-item repeat sample materialization        : AUTHORIZED')
print('Deterministic scoring input materialization   : AUTHORIZED')
print('Condition identity unblinding                 : PROHIBITED')
print('Run aggregation                               : PROHIBITED')
print('Primary endpoint calculation                  : PROHIBITED')
print('Bootstrap / arm comparison                    : PROHIBITED')

print('\\nCELL 7C9 / A003 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C10                          : hybrid single-reviewer evaluation packet materialization only')
print('Human scoring                                 : begins only after Cell 7C10 PASS')
print('Condition unblinding / comparative analysis   : NOT AUTHORIZED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C9
PROTOCOL AMENDMENT A003 — HYBRID SINGLE-REVIEWER EVALUATION DESIGN FREEZE
Notebook                                      : 16_GES_Aware_Genomic_RAG_Cell_7C9_Protocol_Amendment_A003_Hybrid_Single_Reviewer_Evaluation_Freeze.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM CELL 7C8 REVERIFICATION
Cell 7C8 manifest SHA-256                     : e8a2bf79d7c0479063af0fbfb73a77427746c79bdc11600672a82af678b25af7
Cell 7C8 terminal PASS verified               : YES
Cell 7C8 frozen artifacts                     : 11/11 exact hashes + sidecars
Frozen response/review items                  : 1,440
Condition identity opened                     : NO
\nPROTOCOL AMENDMENT A003
Original human-review plan                    : 2 independent reviewers + 